# Data Sources and Provenance

**Manchester City Forecasting Project · Version 1 · September 2026**

This document identifies the project's external data providers and traces the main local files to their sources. The archive contains original downloads, scraper extracts, processed tables, and model outputs. Attribution follows these origins, while the processing notebooks record how the analysis files were constructed.


## 1. External sources

| Key | Provider and reference page | Role in the project |
|---|---|---|
| **FD** | [Football-Data.co.uk — Historical Results and Betting Odds Data](https://www.football-data.co.uk/data.php) | Premier League and Championship results and match statistics; team-strength construction, promotion analysis, and Bayesian updates. |
| **CE** | [ClubElo — Football Club Elo Ratings](https://clubelo.com/) | External comparison for the internally constructed strength index. |
| **FB** | [FBref / Sports Reference — Football Statistics and History](https://fbref.com/en/) | City and Chelsea match logs, plus historical player and team tables collected during development. |
| **FM** | [FotMob — Premier League Player and Team Statistics](https://www.fotmob.com/en-GB/leagues/47/stats/the-premier-league) | Player profiles for the midfield analysis and team-season possession statistics. |
| **US** | [Understat — English Premier League xG Data](https://understat.com/league/EPL) | Expected-goal information and historical match extracts for exploratory analysis. |
| **TM** | [Transfermarkt — Premier League Transfers](https://www.transfermarkt.com/premier-league/transfers/wettbewerb/GB1/) | Summer expenditure and income for exploratory tests of transfer-related prior adjustments. |

ClubElo serves as a validation benchmark; it does not define the operational strength index. Transfer variables were tested but excluded from the final prior specification. The archive also retains exploratory datasets that are not inputs to the deployed score model.


## 2. Source files and derived datasets

Paths below are relative to the supplied data archive. Filename patterns group equivalent files across seasons; they do not imply identical coverage across all datasets.

### Football-Data

The `data_from_footballdata/` directory contains provider CSVs named `E0_YYZZ.csv` and `E1_YYZZ.csv`, covering 2014/15–2025/26. `E0` denotes the Premier League and `E1` the Championship. The five-season modeling tables cover 2021/22–2025/26.

| Local file or group | Lineage |
|---|---|
| `E0_revised_1.csv` | Project revision of the 2025/26 Premier League file, with half-time fields removed. |
| `e0_matches_5seasons.csv` | EPL season files concatenated and standardized for modeling. |
| `processed_e1/e1_matches_5seasons.csv` | Combined Championship season files. |
| `processed_e1/e1_team_match_long_all.csv` | Championship matches reshaped into team-perspective records. |
| `processed_e1/e1_promoted_team_match_long.*` | Records filtered for the promotion analysis. |

Copies under other project directories retain the same upstream attribution to [FD](https://www.football-data.co.uk/data.php). The accompanying notes describe field definitions; the presence of a field, including betting odds, does not imply that it was used in the final model.

### ClubElo

Per-club files in `data_from_Clubelo/`, such as `Man City.csv`, contain locally extracted ClubElo histories [CE](https://clubelo.com/). The acquisition notebook documents page/API retrieval and attempts to recover missing historical coverage. Team names were harmonized with Football-Data naming conventions; the mapping is recorded in `Team_name_standardization.txt`.

### FBref

Within `data_merge_from_FBref/`, the City and Chelsea season folders contain `sportsref_download_*.xls` table exports [FB](https://fbref.com/en/). Fixture, shooting, goalkeeping, and miscellaneous tables were cleaned and joined into season workbooks:

- `Manchester_City_FBref_YYYY_YYYY.xlsx`
- `Chelsea_FBref_YYYY_YYYY.xlsx`

The `fbref_midfield_data/` directory also contains season/statistic CSVs for standard, shooting, passing, passing types, goal creation, defense, possession, and miscellaneous statistics. These were assembled into `EPL_FBref_YYYY-YYYY.xlsx` workbooks. Intermediate merges and variable glossaries are project products derived from the same exports.

### FotMob

The `fotmob_EPL_YYYY-YYYY_raw.csv` files are local player-statistic extracts from FotMob web-data endpoints [FM](https://www.fotmob.com/en-GB/leagues/47/stats/the-premier-league), covering 2023/24–2025/26. Copies appear under `fbref_midfield_data/fotmob_raw/`; that directory name does not change their provider.

The separate file `fotmob_EPL_team_possession_2023_2026.csv` comes from FotMob's **team-season possession endpoint**. The project combines those responses and adds `PossDiff = 2 × Possession − 100`. Its acquisition is separate from the player extracts.

For the midfield bridge, team-season outcomes combine FotMob possession with shot, shot-on-target, and finishing measures calculated from Football-Data match records. That combined analysis therefore draws on both [FM](https://www.fotmob.com/en-GB/leagues/47/stats/the-premier-league) and [FD](https://www.football-data.co.uk/data.php), alongside internally estimated strength controls.

### Understat

The `data_from_understat/` directory contains scraper-generated workbooks, including `manchester_city_understat_matches.xlsx`, `manchester_city_understat_maresca_study.xlsx`, and `chelsea_understat_matches_maresca_study.xlsx`. They are local extractions from Understat EPL season pages [US](https://understat.com/league/EPL), retained for exploratory xG and regime analysis.

### Transfermarkt

The file `data_from_Transfermarkt/EPL_summer_transfers_2014_2025.xlsx` was produced from Transfermarkt summer-window pages [TM](https://www.transfermarkt.com/premier-league/transfers/wettbewerb/GB1/). The acquisition notebook also records a manual correction for Tottenham's 2018/19 entry and its source URL. These data remain part of the provenance record even though transfer adjustments were not retained in the final prior.


## 3. Model-generated files

These files contain project estimates or saved model state. Their documentation should identify the upstream data and estimation method.

| File | Content and origin |
|---|---|
| `team_strength_all_results.xlsx` | Opponent-adjusted team-performance and instability estimates constructed from Football-Data records. |
| `unified_strength_index.xlsx` | Internally constructed scalar strength index, derived from the team-performance estimates and evaluated against ClubElo. |
| `bayesian_match_strength_history.csv` | Pre-match and post-match strengths generated by Bayesian replay using match results and project priors. |
| `bayesian_model_parameters.csv` and `bayesian_preseason_prior_2026_27.csv` | Fitted Bayesian parameters and season-opening priors. |
| `season_state.json` and `season_prediction_results.csv` | Online model state and forecast/result records referenced by the forecasting workflow. |

The last two rows describe outputs referenced by the modeling workflow; their inclusion here is not a claim that every output is bundled in the raw-data archive.


## 4. Acquisition and processing record

The following notebooks document the principal source-to-file transformations. Names are relative to the directory shown.

| Directory | Notebook(s) | Purpose |
|---|---|---|
| `data_from_footballdata/` | `build_match_tables.ipynb` | Combine season CSVs, reshape matches, and construct promotion tables. |
| `data_from_Clubelo/` | `data_scrap_from_clubelo.ipynb` | Extract and align ClubElo histories. |
| `data_merge_from_FBref/` | `merge_fbref_man_city_2324_2425_workbooks.ipynb`; `merge_fbref_man_city_2526_workbooks.ipynb` | Merge City match-log tables. |
| `data_merge_from_FBref/` | `merge_fbref_man_city_2324_2425_fbref_html_fix (1).ipynb` | Handle FBref export parsing during the City merge. |
| `data_merge_from_FBref/` | `merge_fbref_chelsea_2324_2425_2526.ipynb` | Merge Chelsea match-log tables. |
| `data_merge_from_FBref/fbref_midfield_data/` | `Untitled-1.ipynb` | Assemble FBref player workbooks and extract FotMob player statistics. |
| `data_merge_from_FBref/` | `dynamic_system_model_completed.ipynb` | Extract FotMob team possession and build the combined modeling inputs. |
| `data_from_understat/` | `data_scrap_from_understat.ipynb`; `manchester_city_understat_scraper.ipynb`; `chelsea_understat_scraper.ipynb` | Extract Understat match data. |
| `data_from_Transfermarkt/` | `data_scrapping.ipynb` | Extract summer transfer data and document corrections. |


## 5. Attribution and reproducibility

Reports should cite the original provider whenever its data are used, including through processed tables or model inputs. Derived files should also be linked to their processing code. When an analysis combines providers, all relevant sources should be acknowledged.

For example, the City season workbooks can be described as **FBref match tables cleaned and merged within the project**. The filename identifies the local artifact; the provider reference identifies its external origin.

ZIP containers, duplicate copies, temporary extraction folders, caches, and scraper checkpoints do not require separate provider citations. Retaining source snapshots and processing notebooks helps distinguish original records from later transformations.

The date at the top is the documentation date. Dataset retrieval dates should come from actual collection logs where available; they should not be inferred from a filename or replaced by a single document-wide access date. Provider pages and web endpoints may change, so the archived extracts remain the record of the inputs used in this version.
